In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q03/annotated/checkpoints/pre_cell_3.pickle

trying: ['lineitem']


me:  1
trying: ['pd']
me:  0
trying: ['load_customer']
me:  3
trying: ['customer']
me:  3
trying: ['STORAGE_OPTS']
me:  0
trying: ['tpch_parent']
me:  0
trying: ['load_orders']
me:  2
trying: ['load_lineitem']
me:  1
trying: ['DATA_ROOT']
me:  0
trying: ['orders']


me:  2


Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable load_orders
Declaring variable orders
Declaring variable load_customer
Declaring variable customer


In [4]:
%%RecordEvent
%%time
### cell 3 ###

# Optimized for cudf: push filters to GPU with query, drop unnecessary columns early, and chain operations
define_date = '1995-03-04'

# Filter and drop unused columns
fcustomer = customer.query("C_MKTSEGMENT == 'HOUSEHOLD'")[['C_CUSTKEY']]
forders = orders.query(f"O_ORDERDATE < '{define_date}'")[['O_ORDERKEY','O_CUSTKEY','O_ORDERDATE','O_SHIPPRIORITY']]
flineitem = lineitem.query(f"L_SHIPDATE > '{define_date}'")[['L_ORDERKEY','L_EXTENDEDPRICE','L_DISCOUNT']]

# Join and compute TMP
jn2 = (fcustomer
       .merge(forders, on='C_CUSTKEY')
       .merge(flineitem, left_on='O_ORDERKEY', right_on='L_ORDERKEY'))

# Aggregate and sort
total = (jn2.assign(TMP=jn2.L_EXTENDEDPRICE * (1 - jn2.L_DISCOUNT))
           .groupby(['L_ORDERKEY','O_ORDERDATE','O_SHIPPRIORITY'], as_index=False, sort=False)
           .agg({'TMP': 'sum'})
           .sort_values('TMP', ascending=False))

# Final result
res = total[['L_ORDERKEY','TMP','O_ORDERDATE','O_SHIPPRIORITY']]

KeyError: 'C_CUSTKEY'

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q03/rewritten/o4_mini_high/checkpoints/post_cell_3_try_0.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/tpch/notebooks/q03/opt_cell_exec_info_3_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[3], f)


In [ ]:
opt_output = Out.get(4)